In [0]:
df = spark.table("luffy.phase2.plant_growth_data")
display(df.head(10))

Soil_Type,Sunlight_Hours,Water_Frequency,Fertilizer_Type,Temperature,Humidity,Growth_Milestone
loam,5.192294089205035,bi-weekly,chemical,31.719602410244118,61.59186060848997,0
sandy,4.033132702741614,weekly,organic,28.91948412187396,52.42227609891599,1
loam,8.892768570729004,bi-weekly,none,23.179058888285397,44.66053858490323,0
loam,8.241144063085702,bi-weekly,none,18.465886401416917,46.4332272684958,0
sandy,8.374043008245923,bi-weekly,organic,18.12874085342172,63.62592280385192,0
sandy,8.627622080115675,bi-weekly,none,20.004857963291904,67.618726471884,0
loam,4.444267910404542,daily,organic,25.984533294122407,69.57895218629243,1
clay,6.150794371265635,daily,organic,29.291918454001248,69.48090713972769,0
loam,4.695214357150778,bi-weekly,none,28.203947534354626,34.560305152434516,1
loam,9.178620555253561,weekly,organic,20.598677938918858,54.72101523512907,1


In [0]:
 df.count()

193

In [0]:
df.printSchema()

root
 |-- Soil_Type: string (nullable = true)
 |-- Sunlight_Hours: double (nullable = true)
 |-- Water_Frequency: string (nullable = true)
 |-- Fertilizer_Type: string (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Growth_Milestone: integer (nullable = true)



In [0]:
from pyspark.sql.types import StringType

for field in df.schema.fields:
    # Check if the field is a StringType
    if isinstance(field.dataType, StringType):
        print(f"Value counts for column: {field.name}")
        df.groupBy(field.name).count().show(10)

Value counts for column: Soil_Type
+---------+-----+
|Soil_Type|count|
+---------+-----+
|     loam|   62|
|    sandy|   64|
|     clay|   67|
+---------+-----+

Value counts for column: Water_Frequency
+---------------+-----+
|Water_Frequency|count|
+---------------+-----+
|      bi-weekly|   60|
|         weekly|   59|
|          daily|   74|
+---------------+-----+

Value counts for column: Fertilizer_Type
+---------------+-----+
|Fertilizer_Type|count|
+---------------+-----+
|       chemical|   65|
|        organic|   54|
|           none|   74|
+---------------+-----+



In [0]:
from pyspark.ml.feature import StringIndexer

soil_indexer = StringIndexer(inputCol="Soil_Type", outputCol="soil_index")
water_indexer = StringIndexer(inputCol="Water_Frequency", outputCol="water_index")
fert_indexer = StringIndexer(inputCol="Fertilizer_Type", outputCol="fert_index")

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "soil_index",
        "water_index",
        "fert_index",
        "Sunlight_Hours",
        "Temperature",
        "Humidity"
    ],
    outputCol="features"
)

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(featuresCol="features",labelCol="Growth_Milestone")

In [0]:
from pyspark.ml.pipeline import Pipeline

pipeline = Pipeline(stages=[
    soil_indexer,
    water_indexer,
    fert_indexer,
    assembler,
    dt
])


In [0]:
train, test = df.randomSplit([0.7, 0.3],seed=42)

In [0]:
pipeline_model = pipeline.fit(train)

In [0]:
predictions = pipeline_model.transform(test)

In [0]:
display(predictions.select("Growth_Milestone","prediction","probability").head(5))

Growth_Milestone,prediction,probability
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.15384615384615385"",""0.8461538461538461""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6"",""0.4""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0""]}"
0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.15384615384615385"",""0.8461538461538461""]}"
0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0""]}"


In [0]:
predictions.write.format("delta").mode("overwrite").save("/Volumes/luffy/phase2/gold/day_14_predictions")

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="Growth_Milestone", 
    predictionCol="prediction", 
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy: {accuracy:.2%}")

Test Accuracy: 59.26%


In [0]:
import mlflow
mlflow.set_experiment("/Workspace/Users/this15anmoldesu@gmail.com/Databricks 14 Day AI challenge - 02/DAY - 14")
import mlflow.spark

with mlflow.start_run(run_name="PlantGrowth_DecisionTree"):
    # Log parameters
    mlflow.log_param("model_type", "DecisionTreeClassifier")
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # Log the model
    mlflow.spark.log_model(pipeline_model, "model",dfs_tmpdir="/Volumes/luffy/phase2/gold/tmp")

2026/03/05 16:11:56 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/05 16:12:01 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-7df82423-6c09-4079-9220-1a/tmpei08o644/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/05 16:12:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [0]:
```
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Setup MLflow Experiment
experiment_path = "/Workspace/Users/kalyanmistcse@gmail.com/Databricks 14 Day AI Challenge - 02/Day - 14"
mlflow.set_experiment(experiment_path)

# 2. Start Manual Run (No Autologging)
with mlflow.start_run(run_name="PlantGrowth_Manual_RF_Optimized") as run:
    
    # 3. Data Preparation
    df = spark.table("dai.phase2.plant_growth_data")
    train, test = df.randomSplit([0.7, 0.3], seed=42)
    
    # 4. Define Pipeline Stages
    soil_indexer = StringIndexer(inputCol="Soil_Type", outputCol="soil_index", handleInvalid="keep")
    water_indexer = StringIndexer(inputCol="Water_Frequency", outputCol="water_index", handleInvalid="keep")
    fert_indexer = StringIndexer(inputCol="Fertilizer_Type", outputCol="fert_index", handleInvalid="keep")
    
    assembler = VectorAssembler(
        inputCols=["soil_index", "water_index", "fert_index", "Sunlight_Hours", "Temperature", "Humidity"],
        outputCol="features"
    )
    
    # 5. Random Forest with Manual Hyperparameters
    rf_trees = 30
    rf_depth = 10
    
    rf = RandomForestClassifier(
        featuresCol="features", 
        labelCol="Growth_Milestone",
        numTrees=rf_trees,
        maxDepth=rf_depth,
        seed=42
    )
    
    # 6. Fit Pipeline
    pipeline = Pipeline(stages=[soil_indexer, water_indexer, fert_indexer, assembler, rf])
    pipeline_model = pipeline.fit(train)
    
    # 7. Evaluation
    predictions = pipeline_model.transform(test)
    evaluator = MulticlassClassificationEvaluator(
        labelCol="Growth_Milestone", 
        predictionCol="prediction", 
        metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)
    
    # 8. MANUAL MLFLOW LOGGING
    # Log Parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", rf_trees)
    mlflow.log_param("max_depth", rf_depth)
    
    # Log Metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # Log Model with Signature
    input_example = train.limit(5).toPandas()
    mlflow.spark.log_model(
        spark_model=pipeline_model, 
        artifact_path="random_forest_model",
        input_example=input_example,
        dfs_tmpdir="/Volumes/dai/phase2/gold/tmp"
    )

    print(f"Manual Run Successful.")
    print(f"Test Accuracy: {accuracy:.2%}")
    print(f"Run ID: {run.info.run_id}")
```